# Build Olist SQLite Database

## 목적

Olist 원본 CSV를 SQLite 데이터베이스로 적재하여
SQL 분석을 반복 실행할 수 있는 환경을 만든다.

원본 CSV는 변경하지 않고
로컬 SQLite DB를 분석용 데이터 저장소로 사용한다.

In [1]:
# 라이브러리
from pathlib import Path

import sqlite3
import pandas as pd

In [2]:
# Path 설정
project_root = Path.cwd().parent

raw_dir = (
    project_root
    / "data"
    / "raw"
    / "olist"
)

processed_dir = (
    project_root
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

database_path = (
    processed_dir
    / "olist.db"
)

In [3]:
# csv 불러오기
customers_df = pd.read_csv(
    raw_dir
    / "olist_customers_dataset.csv"
)

orders_df = pd.read_csv(
    raw_dir
    / "olist_orders_dataset.csv"
)

order_items_df = pd.read_csv(
    raw_dir
    / "olist_order_items_dataset.csv"
)

products_df = pd.read_csv(
    raw_dir
    / "olist_products_dataset.csv"
)

category_translation_df = pd.read_csv(
    raw_dir
    / "product_category_name_translation.csv"
)

In [4]:
# SQLite 연결
connection = sqlite3.connect(
    database_path
)

In [5]:
# 테이블 저장
customers_df.to_sql(
    "customers",
    connection,
    if_exists="replace",
    index=False
)

orders_df.to_sql(
    "orders",
    connection,
    if_exists="replace",
    index=False
)

order_items_df.to_sql(
    "order_items",
    connection,
    if_exists="replace",
    index=False
)

products_df.to_sql(
    "products",
    connection,
    if_exists="replace",
    index=False
)

category_translation_df.to_sql(
    "category_translation",
    connection,
    if_exists="replace",
    index=False
)

71

In [6]:
# SQL 테이블 확인
table_list_df = pd.read_sql_query(
    """
    SELECT
        name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection
)

print(
    table_list_df
)

                      name
0     analysis_order_items
1          analysis_orders
2     category_translation
3  customer_first_purchase
4                customers
5              order_items
6                   orders
7                 products


In [7]:
# 행 수 검증
table_count_df = pd.read_sql_query(
    """
    SELECT
        'customers' AS table_name,
        COUNT(*) AS row_count
    FROM customers

    UNION ALL

    SELECT
        'orders',
        COUNT(*)
    FROM orders

    UNION ALL

    SELECT
        'order_items',
        COUNT(*)
    FROM order_items

    UNION ALL

    SELECT
        'products',
        COUNT(*)
    FROM products

    UNION ALL

    SELECT
        'category_translation',
        COUNT(*)
    FROM category_translation
    """,
    connection
)

print(
    table_count_df
)

             table_name  row_count
0             customers      99441
1                orders      99441
2           order_items     112650
3              products      32951
4  category_translation         71


In [8]:
connection.close()